# Consolidation: From Dynamic Website to Weekly Digest

## Collecting Bluesky posts responsibly and surfacing what the data science community is talking about

---

**Module:** Unstructured Data (Level 5 module, Level 6 programme)
**Programme:** BSc (Hons) Data Scientist Integrated Degree Apprenticeship
**Session type:** Consolidation tutorial (Topics 1–4, with an NLP preview)

---

### Where this notebook sits in the session

| Segment | What happens | Approx. time |
|:--|:--|:--:|
| 1. The assessment brief | We walk through the brief and the marking criteria together | 20 min |
| 2. **This notebook** | One end-to-end project, from a website that shows us nothing to a digest someone would actually read | 55 min |
| 3. Your formative | What to submit, what useful feedback looks like, common pitfalls | 15 min |

### What we're building

The data team at **Pinnacle Broadband** supports a cohort of data science apprentices. They would like a short weekly digest answering one question: *what is the data science community discussing this week, and what is it sharing?* At the moment, someone skims social media by hand. We will automate that, end to end:

```text
┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐
│ 1. Try the │ → │ 2. Decide  │ → │ 3. Collect │ → │ 4. Prepare │ → │ 5. Find    │ → │ 6. Digest  │
│  website   │   │ the route  │   │ responsibly│   │  the text  │   │  themes    │   │ & evaluate │
└────────────┘   └────────────┘   └────────────┘   └────────────┘   └────────────┘   └────────────┘
```

The techniques are not the point. The **decisions** are, because the decisions are what your technical report is marked on.

### How this notebook is organised

| Part | Mode | What you will see |
|:--|:--:|:--|
| Setup & pre-flight | 🏫 | Dependencies, configuration, an environment check |
| Part 1: Framing the problem | 🏫 | The business question and how we will know if we answered it |
| Part 2: Three ways in | 🏫 | `requests` (fails), Selenium (works, then hits a wall), the AT Protocol API (wins), and a measured appraisal |
| Part 3: Collecting responsibly | 🏫 | Public posts as personal data: label-based exclusion, minimisation, pseudonymisation, a cache |
| Part 4: Preparing the text | 🏫 | Cleaning choices made *for this task*, bot and duplicate control |
| Part 5: Finding themes | 🏫 | TF-IDF + NMF as a demonstration, and choosing the number of themes |
| Part 6: The digest | 🏫 | Turning model output into something a person would read |
| Part 7: Does it work? | 🏫 | An independent check, and the limits of what we built |
| Part 8: Towards your formative | 📝 | The plan behind this project, written as a formative would be |
| Independent study | 📖 | Authenticated access, embeddings, scheduling |

### Learning outcomes addressed

| LO | Description | Where |
|:--|:--|:--|
| LO1 | Perform text analysis and use NLP to extract insights from text data | Parts 4–7 |
| LO2 | Process unstructured and semi-structured data sources and reformat into more machine-readable formats | Parts 2–4 |
| LO3 | Scrape information from external websites and process into a structured format | Parts 2–3 |
| LO4 | Use Python for data manipulation, analysis, visualisation, and system integration | Throughout |
| LO5 | Develop reproducible analysis and robust code | Setup, Parts 3 and 7 |

> 📌 **Not a template to copy.** This notebook shows a *process*. Your project needs your own business challenge, your own data and your own decisions. A submission that reproduces this pipeline on a different hashtag will read as exactly that.

---

# 🏫 Setup & Pre-flight

Install the dependencies first. In your own submission, pin versions in a `requirements.txt` so a marker can rebuild your environment (LO5).

In [ ]:
%pip install --quiet requests beautifulsoup4 pandas selenium nltk scikit-learn matplotlib better-profanity

In [ ]:
import hashlib
import json
import logging
import os
import re
import secrets
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("consolidation")

# House palette for charts
NAVY, STEEL, AMBER = "#1B3A5C", "#2E6DA4", "#D4820A"

### Configuration

Every value that might change between runs lives here, not scattered through the notebook. A reader who wants to reproduce or adapt the work changes one cell.

In [ ]:
# --- Data source -------------------------------------------------------------
SITE_URL = "https://bsky.app"
SEARCH_API = "https://api.bsky.app/xrpc/app.bsky.feed.searchPosts"
HEADERS = {"User-Agent": "BPP-UnstructuredData-teaching/1.0 (educational use)"}

# Search terms: deliberately overlapping, so posts reach us by more than one route
SEARCH_TERMS = [
    '"data science"', "#datascience", '"data scientist"', '"machine learning"',
    "#machinelearning", '"deep learning"', "#dataviz", "#rstats", '"data engineering"',
    '"data analyst"', '"large language model"', '"natural language processing"',
    '"pandas" python', "#statistics", '"AI ethics"',
]
POSTS_PER_TERM = 100          # the maximum one anonymous request returns
REQUEST_DELAY = 0.5           # seconds between API calls: polite, and well under rate limits

# --- Responsible collection ---------------------------------------------------
EXCLUDED_AUTHOR_LABELS = {"!no-unauthenticated", "bot", "spam"}
MAX_POSTS_PER_AUTHOR = 3      # one prolific account should not become a "theme"
WINDOW_DAYS = 7               # the digest describes one week

# --- Caching ------------------------------------------------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
CACHE_PATH = DATA_DIR / "bluesky_ds_posts.jsonl"
CACHE_META = DATA_DIR / "bluesky_ds_posts.meta.json"
REFRESH_CACHE = False         # set True to collect afresh from the live API

# --- Modelling ----------------------------------------------------------------
N_THEMES = 8
RANDOM_STATE = 42

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

### Pre-flight check

Rather than discovering halfway through that something is missing, we check the environment up front and record what is available. The rest of the notebook adapts: if there is no browser, the Selenium cells skip cleanly; if the network is blocked, we use the cached data.

In [ ]:
def preflight() -> dict:
    """Check what this environment can do, and report it as a table."""
    status = {}

    try:
        import selenium  # noqa: F401
        status["selenium installed"] = True
    except ImportError:
        status["selenium installed"] = False

    try:
        r = requests.get(SEARCH_API, params={"q": "data", "limit": 1}, headers=HEADERS, timeout=10)
        status["Bluesky API reachable"] = r.status_code == 200
    except requests.RequestException:
        status["Bluesky API reachable"] = False

    status["cached posts available"] = CACHE_PATH.exists()

    try:
        import nltk
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        from nltk.stem import WordNetLemmatizer
        WordNetLemmatizer().lemmatize("models")
        status["WordNet lemmatiser"] = True
    except Exception:
        status["WordNet lemmatiser"] = False

    print(pd.Series(status, name="available").to_frame().to_string())
    if not (status["Bluesky API reachable"] or status["cached posts available"]):
        print("\n⚠️  No API access and no cache. Ask your tutor for the cached data file "
              f"and place it at {CACHE_PATH}.")
    return status

ENV = preflight()
HAVE_SELENIUM = ENV["selenium installed"]
ONLINE = ENV["Bluesky API reachable"]
HAVE_LEMMATISER = ENV["WordNet lemmatiser"]

---

# 🏫 Part 1: Framing the Problem

*Approximate time: 4 minutes*

We start with the business problem, not the technology. The brief asks you to identify *"a relevant business challenge within your organisation in which the value of the collection and analysis of unstructured data can be clearly shown."* So that comes first.

### The business challenge

Pinnacle Broadband's apprentices are expected to keep up with their field, but the field moves quickly and most of the conversation happens on social media. At the moment, a team lead skims feeds for an hour each week and writes a summary. That process is slow, depends on one person's habits, and covers whatever happened to appear in their timeline.

### Four questions to answer before writing any code

| Question | Our answer |
|:--|:--|
| What is the business question? | What is the data science community discussing this week, and what is it sharing? |
| What data do we need? | Recent public posts about data science: their text, hashtags, links and date |
| What technique answers the question? | Unsupervised theme discovery (we do not know the themes in advance) |
| How will we know it worked? | Themes are distinct and readable; an independent signal (hashtags) agrees with them; the digest can be read in a minute |

That last row matters most. If you can't say how you will judge success, you can't evaluate your work, and evaluation is what lifts a report above description.

> 📌 **Assessment connection:** At 60%+, the LO1 criterion asks for a *"clear connection to the potential value of unstructured data to a business challenge."* At 70%+, it asks you to detail *"the relevance to the apprentice's organisation."* Writing the four answers above *before* you build anything is the simplest way to get there.

---

# 🏫 Part 2: Three Ways In

*Approximate time: 25 minutes*

We will try three routes to the same data, in the order most people would try them, and keep evidence from each. At the end of this part we will *measure* them against one another rather than simply asserting which is best.

### 2.1 Attempt one: `requests` and BeautifulSoup

This is the Topic 3 approach: fetch the page and parse the HTML.

In [ ]:
try:
    response = requests.get(SITE_URL, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")

    print(f"Status code:         {response.status_code}")
    print(f"Page size:           {len(response.text):,} characters")
    print(f"Post elements found: {len(soup.select('[data-testid=postText]'))}")
    print()
    print("All the visible text in the page body:")
    print("   ", soup.body.get_text(" ", strip=True)[:300])
except requests.RequestException as e:
    soup = None
    print(f"Could not reach {SITE_URL} from this network ({type(e).__name__}).")

The request **succeeded**: status 200 and around 28,000 characters of HTML. But there isn't a single post in it. The visible text is a message for browsers that have JavaScript switched off.

This is **client-side rendering**. The server sends an empty shell plus a large JavaScript bundle, and the browser runs that bundle to fetch the posts and build the page. `requests` is an HTTP client, not a browser: it fetches the shell and stops.

Note how this failure looks. Nothing raised an error. A pipeline that checked only the status code would carry on happily with zero rows. This is why we count what we extracted, not just whether the request worked.

Before we reach for a browser, though, it is worth reading what the shell *does* contain.

In [ ]:
if soup is not None:
    comments = [c.strip() for c in soup.find_all(string=lambda s: isinstance(s, Comment))]
    for c in comments:
        if "api" in c.lower():
            print("HTML comment in the page source:", c)

The developers left a note for anyone reading the source: the data is available through an API. Let's also check what the site says about automated access.

### 2.2 Ethics check: `robots.txt`

In [ ]:
from urllib.robotparser import RobotFileParser

try:
    robots_txt = requests.get(f"{SITE_URL}/robots.txt", headers=HEADERS, timeout=10).text
    print(robots_txt)

    rp = RobotFileParser()
    rp.parse(robots_txt.splitlines())
    for path in ["/", "/search?q=data+science", "/intent/compose"]:
        print(f"May crawl {path:<28} {rp.can_fetch('*', SITE_URL + path)}")
except requests.RequestException as e:
    print(f"Could not fetch robots.txt from this network ({type(e).__name__}).")

The rules allow crawling almost everything, and the comment at the top makes a recommendation: if you are considering *bulk or automated crawling*, use the protocol instead.

`robots.txt` tells us what the site operator will tolerate from crawlers. It does **not** tell us whether collecting this data is lawful, what the Terms of Service say, or what we may do with personal data once we have it. We come back to that in Part 3.

> 🔍 **Design decision:** We now have two signals from the operator pointing at the API. A professional would probably go straight there. We will try a browser anyway, because you need to know how to scrape a JavaScript-rendered site, and because seeing *why* it is the weaker route is what gives your appraisal substance.

### 2.3 Attempt two: Selenium

Selenium drives a real browser, so the JavaScript runs and the page is built just as it would be for a person. We then read the finished page.

We use **Selenium Manager** (built into Selenium 4.6+), which finds or downloads the right driver for your installed Chrome. You don't need the separate `webdriver-manager` package, and leaving it out avoids its version-mismatch errors.

In [ ]:
if HAVE_SELENIUM:
    from selenium import webdriver
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait


def make_driver(headless: bool = True):
    """Start Chrome under Selenium. Returns None (and logs why) if no browser is available."""
    if not HAVE_SELENIUM:
        logger.warning("Selenium is not installed; skipping browser steps")
        return None
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1280,2000")
    options.add_argument("--disable-dev-shm-usage")
    if hasattr(os, "geteuid") and os.geteuid() == 0:
        # Chrome refuses to start its security sandbox as root (common in Docker and
        # hosted notebooks). Only in that case do we switch the sandbox off.
        options.add_argument("--no-sandbox")
    if os.environ.get("CHROME_BINARY"):          # only if Chrome is somewhere unusual
        options.binary_location = os.environ["CHROME_BINARY"]
    try:
        return webdriver.Chrome(options=options)
    except WebDriverException as e:
        logger.warning(f"Could not start Chrome: {e.msg.splitlines()[0] if e.msg else e}")
        return None


def selenium_scrape_posts(driver, url: str, scrolls: int = 0, timeout: int = 20) -> pd.DataFrame:
    """Load a Bluesky page in the browser, wait for posts to render, and extract them.

    Args:
        driver:  a live Selenium WebDriver.
        url:     the page to load.
        scrolls: how many times to scroll down to trigger more posts loading.
        timeout: seconds to wait for the first post to appear.

    Returns:
        One row per post: handle, posted, text, likes, permalink. Empty if nothing rendered.
    """
    driver.get(url)
    try:
        # Wait for content rather than sleeping a fixed time: faster when the page is quick,
        # and it does not silently give up when the page is slow.
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid^="feedItem-by-"]'))
        )
    except TimeoutException:
        logger.warning(f"No posts rendered on {url} within {timeout}s")
        return pd.DataFrame()

    for _ in range(scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(2)  # infinite scroll: give the next batch a moment to arrive

    rows = []
    for item in driver.find_elements(By.CSS_SELECTOR, '[data-testid^="feedItem-by-"]'):
        texts = item.find_elements(By.CSS_SELECTOR, '[data-testid="postText"]')
        links = item.find_elements(By.CSS_SELECTOR, 'a[href*="/post/"]')
        likes = item.find_elements(By.CSS_SELECTOR, '[data-testid="likeCount"]')
        rows.append({
            "handle": item.get_attribute("data-testid").removeprefix("feedItem-by-"),
            "posted": links[0].get_attribute("aria-label") if links else None,
            "text": texts[0].text if texts else "",
            "likes": likes[0].text if likes else "0",
            "permalink": links[0].get_attribute("href") if links else None,
        })
    return pd.DataFrame(rows)

A few things worth noticing:

- **The selectors use `data-testid` attributes.** Bluesky's CSS class names are generated automatically and change with each release. The test IDs exist for the developers' own automated tests, so they are the most stable hooks available. They are still not a contract, and they could change tomorrow.
- **Explicit waits, not `time.sleep()`.** We wait until a post exists. The one fixed sleep left, after scrolling, is a known weakness.
- **Graceful failure.** No browser, or nothing rendered? The function logs why and returns an empty table instead of crashing.

Let's point it at the public **Discover** feed.

In [ ]:
driver = make_driver()
selenium_posts = pd.DataFrame()

if driver is not None:
    t0 = time.perf_counter()
    selenium_posts = selenium_scrape_posts(driver, SITE_URL, scrolls=1)
    selenium_seconds = time.perf_counter() - t0
    print(f"Posts extracted: {len(selenium_posts)} in {selenium_seconds:.1f}s")

    mentions_data = selenium_posts["text"].str.contains(
        r"\bdata\b|machine learning|\bAI\b|statistic", case=False, regex=True)
    print(f"Of those, mention data/ML/AI/statistics: {mentions_data.sum()}")
    # We show text and time only. Handles are personal data, and Part 3 deals with them.
    display(selenium_posts[["posted", "text"]].head(8))
else:
    selenium_seconds = None
    mentions_data = pd.Series(dtype=bool)
    print("Browser not available here; the description below explains what you would see.")

It works: Selenium renders the page and we extract structured rows from it. But look at the content. The Discover feed is whatever Bluesky's algorithm chooses to show a logged-out visitor, and almost none of it is about data science. Scrolling gets us more posts, not more *relevant* ones.

To answer our business question, we need to **target**. On a website, targeting means search. So let's search.

In [ ]:
if driver is not None:
    search_url = f"{SITE_URL}/search?q=data+science"
    found = selenium_scrape_posts(driver, search_url, timeout=10)
    page_text = driver.find_element(By.TAG_NAME, "body").text
    print(f"Posts extracted from search: {len(found)}")
    for line in page_text.splitlines():
        if "logged out" in line.lower():
            print("The page says:", line)
    driver.quit()
else:
    print("Browser not available here. Logged out, the search page shows: "
          "'Search is currently unavailable when logged out'.")

**Search is not available to logged-out visitors.** To go further with Selenium, we would have to automate a login. That would mean:

- storing an account password somewhere our code can read it,
- driving the site as a signed-in person, which is a very different thing from reading public pages, and
- doing exactly what `robots.txt` asked automated clients not to do.

> 🔍 **Design decision:** We stop here. Automating a login to get round a restriction the operator has chosen to apply is the point where scraping stops being a technical question and becomes a judgement about whether we *should*. The 80%+ band for LO3 rewards apprentices who identify *"which sources they have decided not to scrape and why."* This is one of those decisions, and it belongs in a technical report.

### 2.4 Attempt three: the AT Protocol API

Bluesky is built on the **AT Protocol**, an open protocol for social data (Kleppmann *et al.*, 2024). Its search endpoint returns posts as JSON, and the first page of results is available without logging in.

In [ ]:
def search_posts(term: str, limit: int = POSTS_PER_TERM, max_retries: int = 3) -> list[dict]:
    """Return up to `limit` recent English-language posts matching `term`.

    Retries with exponential back-off on rate limiting (429) or server errors (5xx).
    Returns an empty list, and logs why, on any other failure.
    """
    params = {"q": term, "lang": "en", "limit": limit, "sort": "latest"}
    for attempt in range(max_retries):
        try:
            r = requests.get(SEARCH_API, params=params, headers=HEADERS, timeout=20)
        except requests.RequestException as e:
            logger.warning(f"{term}: network error ({e}); attempt {attempt + 1}/{max_retries}")
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 200:
            return r.json().get("posts", [])
        if r.status_code == 429 or r.status_code >= 500:
            wait = 2 ** attempt
            logger.warning(f"{term}: HTTP {r.status_code}; retrying in {wait}s")
            time.sleep(wait)
            continue
        logger.error(f"{term}: HTTP {r.status_code}; giving up")
        return []
    return []

In [ ]:
if ONLINE:
    t0 = time.perf_counter()
    sample = search_posts('"data science"')
    api_seconds = time.perf_counter() - t0
    print(f"Posts returned: {len(sample)} in {api_seconds:.1f}s\n")
    example = sample[0]
    print("Top-level fields:   ", list(example.keys()))
    print("Inside 'record':    ", list(example["record"].keys()))
    print("Inside 'author':    ", list(example["author"].keys()))
else:
    sample, api_seconds = [], None
    print("API not reachable from here. Part 3 will load the cached posts instead.")

One request, around a second, up to 100 posts that are actually about our topic. Each post is **semi-structured** JSON: nested objects, optional fields, and lists of varying length. The text sits next to machine-readable metadata a scraper would have to work out from the page: hashtags and links as `facets`, link previews as `embed`, and, as we will see shortly, the author's moderation `labels`.

### 2.5 A measured appraisal

Your report should compare the tools you considered. The strongest comparisons rest on **evidence you gathered**, not adjectives. So let's measure.

In [ ]:
def count_fields(obj, prefix="") -> int:
    """Count leaf fields in a nested JSON object, a rough measure of how much structure we get."""
    if isinstance(obj, dict):
        return sum(count_fields(v, f"{prefix}.{k}") for k, v in obj.items())
    if isinstance(obj, list):
        return count_fields(obj[0], prefix) if obj else 1
    return 1

appraisal = pd.DataFrame([
    {"route": "requests + BeautifulSoup", "posts": 0, "on topic": "n/a",
     "fields per post": 0, "seconds": None, "can target a topic": "No",
     "needs a login to go further": "n/a"},
    {"route": "Selenium (Discover feed)", "posts": len(selenium_posts),
     "on topic": int(mentions_data.sum()) if len(selenium_posts) else "n/a",
     "fields per post": selenium_posts.shape[1] if len(selenium_posts) else 5,
     "seconds": round(selenium_seconds, 1) if selenium_seconds else None,
     "can target a topic": "No (search needs login)", "needs a login to go further": "Yes"},
    {"route": "AT Protocol API", "posts": len(sample),
     "on topic": len(sample) if sample else "n/a",
     "fields per post": count_fields(sample[0]) if sample else None,
     "seconds": round(api_seconds, 1) if api_seconds else None,
     "can target a topic": "Yes", "needs a login to go further": "Only for paging beyond 100"},
]).set_index("route")
appraisal

The verdict is not close. The API is faster, it can be targeted, and it returns far richer structure. It is also the route the operator recommended twice.

So what was Selenium for? It is the right tool when content **only exists after JavaScript runs and no API offers it**: dashboards, single-page apps without public endpoints, flows that need clicks. Knowing when to reach for it, and when not to, is the skill.

One more finding to record. When we tested paging (asking for the *next* 100 results), the anonymous API refused with **403 Forbidden**. Only the first page is open without logging in. We could get round that by searching narrow time windows one after another, but that would be defeating a limit the operator set on purpose. So instead we **widen** our net: one page from each of several related search terms. That gives us breadth instead of depth, and our evaluation will need to say so.

> 📌 **Assessment connection:** The 60%+ band for LO3 asks for *"some discussion of the various tools available"*; 70%+ asks for *"wide discussion"*. A table like this one, with your own measured numbers and a reasoned verdict, is that discussion. Present it as a table, and write *about* it in Section C.

---

# 🏫 Part 3: Collecting Responsibly

*Approximate time: 7 minutes*

Before collecting at scale, we decide what we will keep and what we will not.

### Public is not the same as free to use

A Bluesky handle identifies a person, so a post linked to a handle is **personal data** under UK GDPR, even though anyone can read it. That has consequences:

- We need a **lawful basis**. For internal horizon-scanning, that would most likely be *legitimate interests*, which needs a documented balancing test (ICO, no date).
- We must apply **data minimisation** (UK GDPR Article 5(1)(c)): keep only what the purpose needs. Our digest needs text, tags, links and dates. It does not need names, avatars or account identifiers.
- We should respect what people have **told us** about how they want their data treated.

That last point is easy to miss. Let's look at the author labels in the sample we collected.

In [ ]:
def author_labels(post: dict) -> set[str]:
    """The moderation labels attached to a post's author."""
    return {label["val"] for label in post.get("author", {}).get("labels", [])}

if sample:
    label_counts = Counter(lab for p in sample for lab in author_labels(p))
    print(f"Author labels across {len(sample)} posts:")
    for lab, n in label_counts.most_common():
        print(f"   {lab:<40} {n}")

Three labels matter to us:

| Label | What it means | What we do |
|:--|:--|:--|
| `!no-unauthenticated` | The author has asked Bluesky **not to show their posts to logged-out viewers** | **Exclude.** The website respects this; the raw API still returns their posts. We follow the person's wishes, not the endpoint's behaviour. |
| `bot` | A self-declared automated account | Exclude: our question is what *people* are discussing |
| `spam` | Flagged by Bluesky's moderation | Exclude |

The first one is the most important decision in this notebook, and it was only possible because we looked at the data before using it. It is also a neat example of the **API being more permissive than the website**. Having technical access to data does not mean we have permission to use it.

### Minimise and pseudonymise at the point of collection

We remove what we don't need **before** anything is saved to disk, so it never exists in our files. Handles are replaced with a **salted hash**: the same author always gets the same pseudonym within a run, so we can still limit posts per author, but nobody can reverse it to the handle. The salt is a random value kept in memory and never written down. `@mentions` inside the text are masked for the same reason.

In [ ]:
SALT = os.environ.get("PSEUDONYM_SALT") or secrets.token_hex(16)  # never saved to disk

# @mentions, plus bare Bluesky handles written without the @
MENTION_RE = re.compile(r"@[\w.-]+\.[a-z]{2,}|\b[\w-]+(?:\.[\w-]+)*\.bsky\.social\b", re.IGNORECASE)


def pseudonymise(handle: str, salt: str = SALT) -> str:
    """Replace a handle with a stable, non-reversible pseudonym such as 'author_3f9a1c'."""
    return "author_" + hashlib.sha256((salt + handle).encode()).hexdigest()[:6]


def keep_post(post: dict) -> bool:
    """False if the author carries any label we have chosen to exclude."""
    return not (author_labels(post) & EXCLUDED_AUTHOR_LABELS)


def minimise_post(post: dict, search_term: str) -> dict:
    """Reduce an API post to only the fields our purpose needs."""
    record = post.get("record", {})
    tags, links = [], []
    for facet in record.get("facets", []) or []:
        for feature in facet.get("features", []):
            kind = feature.get("$type", "")
            if kind.endswith("#tag"):
                tags.append(feature.get("tag", "").lower())
            elif kind.endswith("#link"):
                links.append(feature.get("uri"))
    external = (record.get("embed") or {}).get("external") or {}
    if external.get("uri"):
        links.append(external["uri"])

    return {
        "post_id": hashlib.sha256(post["uri"].encode()).hexdigest()[:12],
        "author_id": pseudonymise(post["author"]["handle"]),
        "created_at": record.get("createdAt"),
        "text": MENTION_RE.sub("@user", record.get("text", "")),
        "hashtags": sorted(set(tags)),
        "link_domains": sorted({urlparse(u).netloc.removeprefix("www.") for u in links if u}),
        "likes": post.get("likeCount", 0),
        "reposts": post.get("repostCount", 0),
        "search_term": search_term,
    }

### Tests for the functions we depend on

These functions decide what we keep and what we remove, so a mistake here would be a data-protection failure, not just a bug. A handful of focused tests is worth more than broad coverage (LO5).

In [ ]:
from unittest import TestCase, TestLoader, TextTestRunner

FAKE = {
    "uri": "at://did:plc:abc/app.bsky.feed.post/1",
    "author": {"handle": "someone.bsky.social", "labels": []},
    "record": {"text": "Thanks @friend.bsky.social and pal.bsky.social for the #rstats tip https://example.org/x",
               "createdAt": "2026-10-27T10:00:00Z",
               "facets": [{"features": [{"$type": "app.bsky.richtext.facet#tag", "tag": "rstats"}]},
                          {"features": [{"$type": "app.bsky.richtext.facet#link",
                                         "uri": "https://www.example.org/x"}]}]},
}


class TestResponsibleCollection(TestCase):
    def test_pseudonym_is_stable_and_hides_handle(self):
        a, b = pseudonymise("someone.bsky.social"), pseudonymise("someone.bsky.social")
        self.assertEqual(a, b)
        self.assertNotIn("someone", a)

    def test_different_salt_gives_different_pseudonym(self):
        self.assertNotEqual(pseudonymise("x", "salt1"), pseudonymise("x", "salt2"))

    def test_labelled_authors_are_excluded(self):
        labelled = {**FAKE, "author": {"handle": "h", "labels": [{"val": "!no-unauthenticated"}]}}
        self.assertTrue(keep_post(FAKE))
        self.assertFalse(keep_post(labelled))

    def test_minimised_post_carries_no_handle_or_mention(self):
        row = minimise_post(FAKE, "#rstats")
        self.assertNotIn("someone", json.dumps(row))
        self.assertNotIn("friend", row["text"])
        self.assertNotIn("pal.bsky", row["text"])
        self.assertEqual(row["hashtags"], ["rstats"])
        self.assertEqual(row["link_domains"], ["example.org"])


TextTestRunner(verbosity=2).run(TestLoader().loadTestsFromTestCase(TestResponsibleCollection));

### Collect, with a cache

The live API returns *this week's* posts, so two runs a week apart will never match. To make the analysis reproducible, we save the minimised data once and work from that saved copy afterwards. The cache also lets the notebook run where Bluesky is blocked: your tutor can share the data file.

In [ ]:
def collect_corpus(terms: list[str]) -> tuple[pd.DataFrame, dict]:
    """Query each term once, filter, minimise, and de-duplicate across terms."""
    kept, seen = [], set()
    stats = Counter()
    for term in terms:
        posts = search_posts(term)
        stats["returned"] += len(posts)
        for post in posts:
            if post["uri"] in seen:
                stats["duplicate across terms"] += 1
                continue
            seen.add(post["uri"])
            if not keep_post(post):
                for lab in author_labels(post) & EXCLUDED_AUTHOR_LABELS:
                    stats[f"excluded: {lab}"] += 1
                continue
            kept.append(minimise_post(post, term))
        logger.info(f"{term:<32} returned {len(posts):>3}; kept so far {len(kept)}")
        time.sleep(REQUEST_DELAY)
    stats["kept"] = len(kept)
    return pd.DataFrame(kept), dict(stats)


def load_or_collect() -> tuple[pd.DataFrame, dict]:
    """Use the cache if present (and not refreshing); otherwise collect and save."""
    if CACHE_PATH.exists() and not REFRESH_CACHE:
        meta = json.loads(CACHE_META.read_text()) if CACHE_META.exists() else {}
        logger.info(f"Loading cached posts from {CACHE_PATH} (collected {meta.get('collected_at', '?')})")
        return pd.read_json(CACHE_PATH, lines=True), meta
    if not ONLINE:
        raise RuntimeError(f"No API access and no cache at {CACHE_PATH}.")
    df, stats = collect_corpus(SEARCH_TERMS)
    meta = {"collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "search_terms": SEARCH_TERMS, "excluded_labels": sorted(EXCLUDED_AUTHOR_LABELS),
            "stats": stats}
    df.to_json(CACHE_PATH, orient="records", lines=True)
    CACHE_META.write_text(json.dumps(meta, indent=2))
    return df, meta


posts, meta = load_or_collect()
print(f"Collected at: {meta.get('collected_at')}")
print(json.dumps(meta.get("stats", {}), indent=2))
print(f"\nPosts: {len(posts):,}   Distinct authors: {posts['author_id'].nunique():,}")
posts.head()

Look at the exclusion counts. The `!no-unauthenticated` figure is the number of posts we *could* have used but chose not to. That number belongs in your report, because it shows an ethical decision made *with* data rather than one merely asserted.

> 📌 **Assessment connection:** Section B asks for *anonymised* screenshots at each stage. The table above is safe to screenshot because the minimisation happened in code, before anything was displayed. That is far more reliable than blurring handles by hand afterwards, and it removes the risk of a handle, a username or a file path slipping into your submission.

---

# 🏫 Part 4: Preparing the Text

*Approximate time: 5 minutes*

In Topic 2, we prepared text for **sentiment**, where words like *not* and *very* carry meaning. Here we prepare text for **theme discovery**, where they are noise. Same techniques, different choices, because the downstream task is different. Saying that explicitly in your report is the difference between describing your pipeline and justifying it.

### First, look at the data quality

In [ ]:
posts["created_at"] = pd.to_datetime(posts["created_at"], utc=True, format="ISO8601", errors="coerce")
posts["word_count"] = posts["text"].str.split().str.len()

per_author = posts["author_id"].value_counts()
print(f"Date range:              {posts['created_at'].min():%d %b %Y} to {posts['created_at'].max():%d %b %Y}")
print(f"Median words per post:   {posts['word_count'].median():.0f}")
print(f"Exact duplicate texts:   {posts['text'].duplicated().sum()}")
print(f"Busiest author's posts:  {per_author.iloc[0]}  ({per_author.iloc[0] / len(posts):.0%} of the corpus)")
print(f"Authors with 10+ posts:  {(per_author >= 10).sum()}")
print()
print("Oldest post returned for each search term:")
print(posts.groupby("search_term")["created_at"].min().dt.strftime("%d %b").sort_values().to_string())

Four things stand out:

1. **The time span is uneven.** Each term returns its latest 100 posts. Busy terms fill that in a day or two, while quiet ones reach back weeks. A *weekly* digest needs a fixed window.
2. **Posts are short.** The median is around 30 words, which shapes the choice of model in Part 5.
3. **There are repeats**, mostly the same announcement posted again and again.
4. **A few accounts dominate.** Some unlabelled automated accounts (job boards, course promoters, news relays) post dozens of times. If we leave them in, a single account becomes a "theme".

There is a fifth problem that a summary won't show you: we asked for English (`lang=en`), but **language tags are chosen by the author**, so some Japanese, Russian and Spanish posts arrive labelled as English.

### The cleaning function

In [ ]:
from better_profanity import profanity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

if HAVE_LEMMATISER:
    from nltk.stem import WordNetLemmatizer
    _lemmatiser = WordNetLemmatizer()

# Words that are common in this corpus but say nothing about *which* theme a post belongs to
DOMAIN_STOP_WORDS = {
    "amp", "just", "like", "new", "really", "think", "know", "way", "use", "using", "make",
    "got", "want", "need", "good", "time", "people", "thing", "today", "day", "week", "year",
    "does", "did", "doing", "say", "said", "yes", "lot", "let", "going", "look", "thanks",
}
STOP_WORDS = set(ENGLISH_STOP_WORDS) | DOMAIN_STOP_WORDS

URL_RE = re.compile(r"https?://\S+|\b[\w-]+(?:\.[\w-]+)+/\S*")


def clean_for_themes(text: str) -> str:
    """Reduce a post to the content words that signal its theme.

    Steps: lower-case; remove URLs and masked mentions; keep hashtag words (#rstats -> rstats);
    keep letters only; drop stop words and very short tokens; lemmatise (models -> model).
    Returns a space-separated string, which is what scikit-learn's vectorisers expect.
    """
    text = URL_RE.sub(" ", text.lower())
    text = text.replace("@user", " ").replace("#", " ")
    tokens = re.findall(r"[a-z]{3,}", text)
    tokens = [t for t in tokens if t not in STOP_WORDS]
    if HAVE_LEMMATISER:
        tokens = [_lemmatiser.lemmatize(t) for t in tokens]
    return " ".join(tokens)


posts["clean"] = posts["text"].apply(clean_for_themes)
for _, row in posts.sample(3, random_state=RANDOM_STATE).iterrows():
    print(f"BEFORE: {row['text'][:110]!r}")
    print(f"AFTER:  {row['clean'][:110]!r}\n")

### From posts to documents: one filter at a time

Each filter answers one of the problems above. We record the count after every step, so the effect of each decision is visible. A table like this makes a good Section B screenshot.

In [ ]:
def mostly_latin(text: str, threshold: float = 0.7) -> bool:
    """True if most letters are in the basic Latin alphabet: a cheap check on self-declared language tags."""
    letters = [c for c in text if c.isalpha()]
    return not letters or sum(c.isascii() for c in letters) / len(letters) >= threshold


window_start = posts["created_at"].max() - pd.Timedelta(days=WINDOW_DAYS)
steps = [("collected and filtered (Part 3)", posts)]
d = posts[posts["created_at"] >= window_start];                 steps.append((f"within the last {WINDOW_DAYS} days", d))
d = d[d["text"].apply(mostly_latin)];                            steps.append(("mostly Latin script", d))
d = d[~d["clean"].duplicated()];                                 steps.append(("repeats removed", d))
d = d[d["clean"].str.split().str.len() >= 5];                    steps.append(("at least 5 content words", d))
d = (d.sort_values("created_at", ascending=False)
      .groupby("author_id").head(MAX_POSTS_PER_AUTHOR));         steps.append((f"at most {MAX_POSTS_PER_AUTHOR} per author", d))
docs = d.reset_index(drop=True)

funnel = pd.DataFrame([{"step": name, "posts": len(x), "authors": x["author_id"].nunique()}
                       for name, x in steps]).set_index("step")
funnel

> 🔍 **Design decision:** Capping each author at three posts is a value judgement, and it should be presented as one. It means the themes represent *what many people are discussing*, not *what is posted most often*. A different business question (for example, "which accounts should our apprentices follow?") would need the opposite choice. The script check is a heuristic too: it removes Japanese and Russian text, but Latin-script languages such as Spanish or Tagalog still get through. That goes in the limitations.

> 📌 **Assessment connection:** *"I removed duplicates"* is descriptive. Here is the analytical version, written about the data used to build this session: *"Because the anonymous API returns the latest 100 posts per term, quieter terms reached back several weeks, so the corpus was restricted to the most recent seven days. Eleven accounts had posted ten or more times; capping each author at three posts means the themes reflect breadth of discussion rather than posting volume."* It states the evidence, the decision and the reason.

---

# 🏫 Part 5: Finding Themes

*Approximate time: 8 minutes*

This is a **demonstration**, not a lesson in topic modelling: we will meet these ideas properly in later topics. The aim here is to show an NLP technique producing business value, and the decisions that surround it.

### The approach in three sentences

1. **TF-IDF** turns each post into a vector of word weights, where a word scores highly if it is frequent in *this* post but rare across the corpus.
2. **Non-negative Matrix Factorisation (NMF)** splits that document-by-word table into two smaller ones: *themes × words* (what each theme is about) and *documents × themes* (how strongly each post belongs to each theme) (Lee and Seung, 1999).
3. We read the words that define each theme and decide what, if anything, it means.

> 🔍 **Design decision: why NMF rather than LDA?** LDA, which you may have seen elsewhere, models word *counts* and needs a reasonable amount of text per document to estimate its mixtures. Our posts have a median of about 30 words. NMF on TF-IDF weights tends to produce more coherent, readable topics on short texts (O'Callaghan *et al.*, 2015), it is fast, and it is deterministic for a given input and initialisation. That is a trade-off rather than a rule: on long documents, LDA's probabilistic output can be the better choice.

### Build the TF-IDF matrix

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

vectoriser = TfidfVectorizer(min_df=4, max_df=0.3, ngram_range=(1, 2))
X = vectoriser.fit_transform(docs["clean"])

# Guard the vocabulary: offensive words must never be able to become a theme label on screen.
# We drop them from the *vocabulary*, not the posts, so the posts still count.
blocked = [w for w in vectoriser.get_feature_names_out() if profanity.contains_profanity(w)]
if blocked:
    vectoriser = TfidfVectorizer(min_df=4, max_df=0.3, ngram_range=(1, 2), stop_words=blocked)
    X = vectoriser.fit_transform(docs["clean"])
terms = vectoriser.get_feature_names_out()

print(f"Documents × vocabulary: {X.shape[0]:,} × {X.shape[1]:,}  (blocked terms: {len(blocked)})")

`min_df=4` ignores words used in fewer than four posts (too rare to define a theme). `max_df=0.3` ignores words in more than 30% of posts (too common to tell themes apart). `ngram_range=(1, 2)` keeps pairs such as *machine learning*, which mean something different from *machine* and *learning* separately. All three are settings you should say you experimented with.

### How many themes?

There is no single correct number. We can, however, check the most important property: whether each theme's top words actually **occur together** in the posts. This is **topic coherence**, measured here with normalised pointwise mutual information (NPMI): around 0 means the words co-occur only as often as chance would predict, and higher is better (Bouma, 2009; Röder, Both and Hinneburg, 2015).

In [ ]:
def npmi_coherence(model, X_binary, top_n: int = 8) -> float:
    """Mean NPMI over all pairs of each theme's top words, averaged across themes."""
    n_docs = X_binary.shape[0]
    doc_freq = np.asarray(X_binary.sum(axis=0)).ravel()
    scores = []
    for weights in model.components_:
        top = weights.argsort()[::-1][:top_n]
        for i, a in enumerate(top):
            for b in top[i + 1:]:
                both = X_binary[:, a].multiply(X_binary[:, b]).sum()
                if both == 0:
                    scores.append(-1.0)
                    continue
                p_ab = both / n_docs
                pmi = np.log(p_ab / ((doc_freq[a] / n_docs) * (doc_freq[b] / n_docs)))
                scores.append(pmi / -np.log(p_ab))
    return float(np.mean(scores))


X_binary = (X > 0).astype(int).tocsc()
sweep = []
for k in range(4, 15):
    m = NMF(n_components=k, init="nndsvda", random_state=RANDOM_STATE, max_iter=600).fit(X)
    sweep.append({"themes": k, "coherence": npmi_coherence(m, X_binary)})
sweep = pd.DataFrame(sweep).set_index("themes")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(sweep.index, sweep["coherence"], marker="o", color=STEEL)
ax.axvline(N_THEMES, color=AMBER, linestyle="--", label=f"chosen: {N_THEMES}")
ax.set_xlabel("Number of themes")
ax.set_ylabel("Mean NPMI coherence")
ax.set_title("Topic coherence by number of themes")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

Look for the region where coherence is high before it falls away as the themes become too narrow to mean anything. On the data used to build this session, it peaked at seven and stayed close to that at eight. Eight themes is still few enough to read in a weekly digest. The metric narrows the choice, and readability for the audience settles it. If the chart points somewhere else for *this week's* data, change `N_THEMES` in the configuration cell, and say in your report which value you chose and why.

### Fit the model and read the themes

In [ ]:
nmf = NMF(n_components=N_THEMES, init="nndsvda", random_state=RANDOM_STATE, max_iter=600)
W = nmf.fit_transform(X)                        # documents × themes
docs["theme"] = W.argmax(axis=1)
docs["theme_strength"] = W.max(axis=1)


def theme_table(model, n_words: int = 8) -> pd.DataFrame:
    rows = []
    for t, weights in enumerate(model.components_):
        top = [terms[i] for i in weights.argsort()[::-1][:n_words]]
        rows.append({"theme": t, "posts": int((docs["theme"] == t).sum()),
                     "provisional label": " / ".join(top[:3]), "top words": ", ".join(top)})
    out = pd.DataFrame(rows).set_index("theme")
    out["share"] = (out["posts"] / out["posts"].sum()).map("{:.0%}".format)
    return out


themes = theme_table(nmf)
themes

Before reading on, look at the table and ask yourself which themes you could name in a few words, and which you couldn't.

On the data used to build this session, the themes included *AI agents*, *R and data visualisation*, *large language models*, *machine and deep learning research*, *data science with Python (with a good deal of course advertising)*, and *remote data jobs*. One theme collected posts about a television pundit described as a "data analyst". That is real **query drift**: the search term matched, but the meaning did not. Your live run may differ, because this is this week's conversation.

Note that the labels are **provisional and machine-generated** (the top three words). Naming a theme is interpretation, not output. Names chosen by a person are not stable across runs or weeks, which is why we don't hard-code them.

In [ ]:
counts = themes["posts"].sort_values()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh([themes.loc[t, "provisional label"] for t in counts.index], counts.values, color=NAVY)
for y, v in enumerate(counts.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=9)
ax.set_xlabel("Posts (dominant theme)")
ax.set_title("What the data science community posted about this week")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

> 📌 **Assessment connection:** A chart earns its place in Section B when it has a title that states the finding, labelled axes, and a sentence in Section C saying what it shows *and what it doesn't*. This one shows relative volume within our sample. It does **not** show what is popular on Bluesky overall, because our sample was shaped by our search terms.

---

# 🏫 Part 6: The Digest

*Approximate time: 5 minutes*

A topic model is not a deliverable. A team lead does not want a matrix; they want something to read on a Monday morning. This is the step most submissions leave out, and it is where the business value shows.

For each theme, we include the most representative posts (the highest theme strength, shortened, with authors pseudonymised), the hashtags people used, and the sites they linked to.

In [ ]:
SEARCH_HASHTAGS = {t.lstrip("#").lower() for t in SEARCH_TERMS if t.startswith("#")}
SOCIAL_DOMAINS = {"bsky.app", "bsky.social", "x.com", "twitter.com", "youtu.be", "youtube.com",
                  "linkedin.com", "lnkd.in", "buff.ly", "bit.ly", "dlvr.it", "ift.tt"}


def representative_posts(members: pd.DataFrame, n: int = 2) -> pd.DataFrame:
    """Pick readable, typical posts: strong theme membership, weighted by engagement.

    Ranking by theme strength alone picks out lists of hashtags, which match a theme's
    words perfectly and say nothing. So we skip posts that are mostly hashtags or very
    short, give a modest boost to posts other people engaged with, and take one per author.
    """
    words = members["text"].str.split().str.len().clip(lower=1)
    hashtag_share = members["text"].str.count(r"#\w+") / words
    pool = members[(words >= 12) & (hashtag_share < 0.3)]
    if len(pool) < n:
        pool = members
    score = pool["theme_strength"] * (1 + np.log1p(pool["likes"] + pool["reposts"]))
    return (pool.assign(score=score).sort_values("score", ascending=False)
                .drop_duplicates("author_id").head(n))


def excerpt(text: str, length: int = 160) -> str:
    """One-line, shortened, with offensive words masked: this text goes in front of people."""
    return profanity.censor(" ".join(text.split())[:length])


def theme_digest_entry(t: int, n_posts: int = 2) -> dict:
    members = docs[docs["theme"] == t]
    top_posts = representative_posts(members, n_posts)
    # Count each hashtag or site once per author, and keep only those used by 2+ people:
    # one person's tag is not a trend, and this keeps one-off oddities out of the digest.
    tag_authors, site_authors = Counter(), Counter()
    for author, group in members.groupby("author_id"):
        tag_authors.update({t for ts in group["hashtags"] for t in ts
                            if t not in SEARCH_HASHTAGS and t.isascii() and t.isalnum()})
        site_authors.update({d for ds in group["link_domains"] for d in ds if d not in SOCIAL_DOMAINS})
    tags = Counter({t: n for t, n in tag_authors.items()
                    if n >= 2 and not profanity.contains_profanity(t)})
    sites = Counter({d: n for d, n in site_authors.items() if n >= 2})
    return {
        "label": themes.loc[t, "provisional label"],
        "posts": len(members),
        "share": themes.loc[t, "share"],
        "examples": [(r.author_id, excerpt(r.text)) for r in top_posts.itertuples()],
        "hashtags": [f"#{h}" for h, _ in tags.most_common(4)],
        "sites": [s for s, _ in sites.most_common(3)],
    }


def build_digest() -> str:
    period = f"{docs['created_at'].min():%d %b} – {docs['created_at'].max():%d %b %Y}"
    lines = [f"## Data science on Bluesky: weekly digest ({period})",
             f"*{len(docs):,} posts from {docs['author_id'].nunique():,} people, "
             f"grouped into {N_THEMES} themes. Authors pseudonymised.*", ""]
    for t in themes.sort_values("posts", ascending=False).index:
        e = theme_digest_entry(t)
        lines.append(f"### {e['label']}  ·  {e['posts']} posts ({e['share']})")
        for author, text in e["examples"]:
            lines.append(f"> {text}… — *{author}*")
            lines.append("")
        if e["hashtags"]:
            lines.append(f"**Also tagged:** {', '.join(e['hashtags'])}  ")
        if e["sites"]:
            lines.append(f"**Most-linked sites:** {', '.join(e['sites'])}")
        lines.append("")
    return "\n".join(lines)


from IPython.display import Markdown, display

digest = build_digest()
digest_path = OUTPUT_DIR / f"digest_{datetime.now():%Y-%m-%d}.md"
digest_path.write_text(digest, encoding="utf-8")
display(Markdown(digest))
print(f"Saved to {digest_path}")

Choosing the example posts was a design problem in its own right. The first version simply took the posts with the highest theme strength. Those turned out to be lists of hashtags, which match a theme's words perfectly and say nothing to a reader. Being able to explain a fix like that, and why it was needed, is good Section C material.

Look at the *Most-linked sites* lines too. In some themes the most-linked sites are training providers advertising courses. That is a finding in its own right: a noticeable share of what gets tagged "data science" is promotion, not discussion.

This could be forwarded as it is. A real version would still have a person read it before it went out: to rename the themes, drop any that drifted (such as the pundit), and check that no example post is something an apprentice should not be pointed to. The model **triages**; a person **edits**.

> 🔍 **Design decision:** The digest shows excerpts and pseudonyms, not handles or links to posts. Linking to the original posts would be more useful to readers, but it would reintroduce identifiable personal data. That would need the legitimate-interests balancing test mentioned in Part 3, completed and documented. A trade-off like this, between usefulness and privacy, is exactly what Section C should discuss.

---

# 🏫 Part 7: Does It Work?

*Approximate time: 6 minutes*

An unsupervised model has no accuracy score. There are no correct labels to compare against. What we *can* do is test whether the model agrees with an **independent signal** it never saw.

Hashtags are such a signal: they are chosen by the authors, and our model never used them as labels. If the themes are real, two posts in the **same** theme should share hashtags more often than two posts in **different** themes. We exclude the hashtags we searched for, because every post found by `#rstats` has that tag, and counting them would be circular.

In [ ]:
def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if (a or b) else 0.0


tagged = docs.assign(tagset=docs["hashtags"].apply(lambda h: set(h) - SEARCH_HASHTAGS))
tagged = tagged[tagged["tagset"].str.len() > 0].reset_index(drop=True)

rng = np.random.default_rng(RANDOM_STATE)
pairs = rng.integers(0, len(tagged), size=(20_000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
same = tagged["theme"].values[pairs[:, 0]] == tagged["theme"].values[pairs[:, 1]]
overlap = np.array([jaccard(tagged.at[i, "tagset"], tagged.at[j, "tagset"]) for i, j in pairs])

result = pd.Series({"same theme": overlap[same].mean(), "different themes": overlap[~same].mean()})
print(f"Posts with non-search hashtags: {len(tagged)}")
print(f"Mean hashtag overlap, same theme:       {result['same theme']:.3f}")
print(f"Mean hashtag overlap, different themes: {result['different themes']:.3f}")
print(f"Ratio: {result['same theme'] / max(result['different themes'], 1e-9):.1f}×")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
bars = ax.bar(result.index, result.values, color=[NAVY, STEEL], width=0.5)
ax.bar_label(bars, fmt="%.3f")
ax.set_ylabel("Mean Jaccard overlap of hashtags")
ax.set_title("Posts in the same theme share more hashtags")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Posts in the same theme share noticeably more hashtags than posts in different themes. The model has found structure that the authors' own tagging independently supports. The absolute numbers are small because most pairs of posts share no hashtags at all. The **ratio** is the finding.

### Strengths and bounded limitations

**What this pipeline does well:** it collects targeted data in about 15 requests; it respects both the operator's guidance and individual authors' wishes; it removes personal data before anything is saved; and it turns model output into something a person would read, with an independent check that the themes are real.

**What it does not do, stated precisely:**

| Limitation | Consequence |
|:--|:--|
| Anonymous search returns only the latest 100 results per term | Busy terms cover a day or two, quiet ones several weeks. Within our seven-day window, busy terms are under-sampled and the mix is shaped by our choice of terms |
| Language tags are self-declared | The script check removes non-Latin text, but other Latin-script languages (Spanish, Tagalog) can get through, and non-English discussion of the same topics is invisible |
| Moderation labels are incomplete | Some automated accounts are unlabelled. The author cap limits their effect but does not remove them |
| Search terms are ambiguous | Query drift (the pundit theme). An earlier draft used `#nlp`, which also matches neuro-linguistic programming and unrelated adult content, so it was replaced |
| Live data changes weekly | The analysis is reproducible only from the cache. Re-running against the API next week gives a different, equally valid, result |
| Bag-of-words | *"LLMs are great"* and *"LLMs are not great"* look the same to this model |

**What we would do next:** use authenticated access for full-week coverage (see Independent Study); try sentence embeddings for themes based on meaning rather than shared words; and run the pipeline weekly, comparing themes over time.

> 📌 **Assessment connection:** Look at the 80–100% descriptors. The word *bounded* appears again and again: *"The processing pipeline is bounded"*, *"Reproducibility is bounded"*. A table like the one above, where each limitation is stated with its specific consequence, is what *bounded* looks like in practice. Compare *"there are some limitations with the data"* with *"anonymous search returns 100 results per term, so the sample covers roughly the last few days."*

---

# 📝 Part 8: Towards Your Formative

*For the final part of the session*

The brief asks for your formative to be *"a draft / bullet point framework of your intended Summative Submission"*. A high-level plan is enough. Here is the plan behind this notebook, written the way a formative could be. Read it as an example of **level of detail**, not as content to reuse.

**Business challenge:** apprentices need a weekly view of their field; the current manual skim is slow and depends on one person.

**Data source and route:** public Bluesky posts. The website is client-side rendered, and search requires a login. The AT Protocol API offers anonymous search (first page only). Rejected: Selenium with automated login (credentials, operator guidance, ToS).

**Legal and ethical position:** posts are personal data; the likely lawful basis is legitimate interests; minimisation at collection; exclude authors labelled `!no-unauthenticated`, `bot` or `spam`; pseudonymise; no handles in outputs.

**Processing:** JSON to DataFrame; extract hashtags and links from facets; de-duplicate; cap per author; clean text for theme discovery.

**Analysis:** TF-IDF with NMF; choose the number of themes using NPMI coherence and readability; compare against LDA.

**Output:** a weekly digest showing themes, example excerpts, hashtags and sites.

**Evaluation:** hashtag agreement within themes against a between-theme baseline; manual review of themes.

**Risks and limits:** first-page-only coverage, query drift, residual bots, live data changing weekly.

**Reproducibility:** a configuration cell, a pre-flight check, a cache with metadata, tests for the data-protection functions, and pinned requirements.

**Questions for my tutor:** Is the per-author cap defensible? Is hashtag agreement enough evaluation?

That last line is the most valuable part of a formative. **Ask the questions you actually want answered.** Feedback is far more useful when it is aimed at a decision you are unsure about.

### Before you submit anything

- [ ] No names, handles, employee IDs, customer data or internal URLs in code, output or screenshots
- [ ] No personal file paths such as `C:\Users\<your name>\...` visible in screenshots
- [ ] Only your SRN identifies you
- [ ] You can explain why you chose each tool, and what you rejected

---

# 📖 Independent Study

### Authenticated access

Logging in with an **app password** (created under Settings → Privacy and security → App passwords, never your main password) unlocks cursor-based paging, so you can collect a full week or more. Three rules apply:

1. Keep credentials **out of your notebook**. Read them from environment variables, and never commit them or show them in screenshots.
2. Use an established client such as the `atproto` Python package rather than hand-building authentication.
3. Being logged in does not change your obligations. The same personal-data reasoning applies, and the exclusions in Part 3 still hold.

```python
# Sketch only: run in your own environment
# pip install atproto
import os
from atproto import Client

client = Client()
client.login(os.environ["BSKY_HANDLE"], os.environ["BSKY_APP_PASSWORD"])
page = client.app.bsky.feed.search_posts({"q": "data science", "lang": "en", "limit": 100})
next_page = client.app.bsky.feed.search_posts(
    {"q": "data science", "lang": "en", "limit": 100, "cursor": page.cursor})
```

### Extensions

1. **Themes by meaning.** Replace TF-IDF with sentence embeddings, then cluster them. Does the pundit theme disappear? Do any new problems appear?
2. **Week on week.** Run the collection on two different weeks and match the themes between runs. Which themes persist, and which are new?
3. **What is being shared?** Rank the linked domains across the whole corpus. What proportion are papers, code repositories, news or courses? Classifying them is a small NLP problem in its own right.
4. **Sensitivity.** Change `MAX_POSTS_PER_AUTHOR` to 1 and to 10. How much do the themes change? What does that tell you about your result?

---

# ✅ Reflection

1. **Start with the question.** The business need decided which data, which technique and which evaluation we used.
2. **Try the obvious route, and keep the evidence.** A failed `requests` call and a login wall were the strongest parts of the appraisal.
3. **Technical access is not permission.** The API returned posts whose authors had asked not to be shown to logged-out viewers. We left them out.
4. **Minimise early.** Data you never saved can't leak into a screenshot.
5. **A model is not a deliverable.** The digest is.
6. **State limits precisely.** Bounded claims score higher than either confident ones or vague ones.

### Confidence check

Rate your confidence (1 = not at all, 5 = very confident) after this session:

| LO | Description | Confidence (1–5) |
|:--|:--|:--:|
| LO1 | Perform text analysis and use NLP to extract insights from text data | |
| LO2 | Process unstructured and semi-structured data sources and reformat into more machine-readable formats | |
| LO3 | Scrape information from external websites and process into a structured format | |
| LO4 | Use Python for data manipulation, analysis, visualisation, and system integration | |
| LO5 | Develop reproducible analysis and robust code | |

### References

Bouma, G. (2009) 'Normalized (pointwise) mutual information in collocation extraction', *Proceedings of the Biennial GSCL Conference 2009*, pp. 31–40.

Information Commissioner's Office (no date) *Legitimate interests*. Available at: https://ico.org.uk/for-organisations/uk-gdpr-guidance-and-resources/lawful-basis/legitimate-interests/ (Accessed: 28 October 2026).

Kleppmann, M., Frazee, P., Gold, J., Graber, J., Holmgren, D., Ivy, D., Johnson, J., Newbold, B. and Volpert, J. (2024) 'Bluesky and the AT Protocol: usable decentralized social media', *Proceedings of the ACM Conext-2024 Workshop on the Decentralization of the Internet*, pp. 1–7.

Lee, D.D. and Seung, H.S. (1999) 'Learning the parts of objects by non-negative matrix factorization', *Nature*, 401, pp. 788–791.

O'Callaghan, D., Greene, D., Carthy, J. and Cunningham, P. (2015) 'An analysis of the coherence of descriptors in topic modeling', *Expert Systems with Applications*, 42(13), pp. 5645–5657.

Röder, M., Both, A. and Hinneburg, A. (2015) 'Exploring the space of topic coherence measures', *Proceedings of the Eighth ACM International Conference on Web Search and Data Mining*, pp. 399–408.

*Documentation:* AT Protocol, https://atproto.com · Bluesky API, https://docs.bsky.app · Selenium, https://www.selenium.dev/documentation/ · scikit-learn NMF, https://scikit-learn.org/stable/modules/decomposition.html#nmf

---

*Consolidation tutorial developed for the BPP University BSc (Hons) Data Scientist programme. For educational use.*